# Company Quarter Prompt Search

In [1]:
import sys
from pathlib import Path

project_root = next(
    (
        path
        for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
        if (path / "src" / "data_collection" / "consts.py").is_file()
    ),
    None,
)
if project_root is None:
    raise RuntimeError("Could not locate project root containing 'src' directory.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import torch
from IPython.display import display
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM

from src.data_collection.consts import DB_PARAMS
from src.data_collection.model_driver.model_driver_class import ModelDriver
from src.data_analysis.data_fetcher.data_fetcher_class import DataFetcher

/home/maxim-shibanov/anaconda3/envs/vllm_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
fq_verbolizer = {
    "first": ["first", "january", "february", "march", "spring", "earlyyear"],
    "second": ["second", "april", "may", "june", "midyear"],
    "third": ["third", "july", "august", "september", "lateyear"],
    "fourth": ["fourth", "october", "november", "december", "yearend", "annual", "yearly"],
}

quarter_prompts = [
    "Read the report excerpt and identify the calendar quarter of the SEC filing date. Selected quarter:",
    "Classify the calendar quarter of the filing date for this SEC report. Classification:",
    "You are tagging SEC filings by submission quarter. From this report excerpt, the filing belongs to the",
    "Imagine you are reviewing SEC metadata and need to assign the filing to a calendar quarter. The correct quarter is",
    "Treat this as a metadata classification task for an SEC document. Your goal is not to identify the reporting period quarter, but the actual calendar quarter in which the filing was submitted to the SEC. Based on the excerpt, the filing quarter should be labeled:",
    "Read the report excerpt and identify this company's quarter within its reporting year. Selected company quarter:",
    "Classify the company quarter rank for this SEC report within the company year. Classification:",
    "You are tagging company reports by quarter rank within the company year. From this report excerpt, the company quarter is",
    "Imagine you are reviewing company reporting metadata and need to assign the report to the correct company quarter within the year. The correct company quarter is",
    "Treat this as a metadata classification task for an SEC document. Your goal is to identify the company's quarter rank within its reporting year. Based on the excerpt, the company quarter should be labeled:",
    "Using the excerpt, infer where this report falls in the company's own yearly reporting sequence. Choose among first, second, third, and fourth. The company quarter rank is:",
    "Ignore the SEC submission month and focus on the company's reporting cycle. Which company quarter does this report represent within the year? Answer:",
    "Determine the quarter number of this report within the company's annual reporting pattern, rather than the calendar filing quarter. The correct company quarter is:",
    "Based on the reporting language and report structure, assign this filing to the company's first, second, third, or fourth reporting quarter of the year. Company quarter:",
    "You are labeling reports by quarter order within each company's reporting year. From this excerpt, the most likely company quarter rank is:",
]

In [3]:
model_name = "mistralai/Mistral-7B-v0.1"
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    max_memory={0: "12GiB", "cpu": "64GiB"},
)

model_driver = ModelDriver(model_name=model_name, model=model)

Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  2.13it/s]
2026-05-24 17:26:45,239 [WARNING] Some parameters are on the meta device because they were offloaded to the cpu.


In [4]:
quarter_name_map = {1: "first", 2: "second", 3: "third", 4: "fourth"}

fetcher = DataFetcher(DB_PARAMS)
reports_df = fetcher._fetch_reports(regressors=["raw_text"])
reports_df = reports_df[reports_df["report_type"].isin(["10-K", "10-Q"])].copy()
reports_df["filed_date"] = pd.to_datetime(reports_df["filed_date"])
reports_df = reports_df.sort_values("filed_date").copy()
reports_df["year"] = reports_df["filed_date"].dt.year
reports_df["filing_rank"] = reports_df.groupby(["cik", "year"]).cumcount()
reports_df = reports_df[
    reports_df["filing_rank"] >= (
        reports_df.groupby(["cik", "year"])["filing_rank"].transform("max") - 3
    )
].copy()
reports_df["true_label"] = (reports_df.groupby(["cik", "year"]).cumcount() + 1).map(quarter_name_map)

sample_q1 = reports_df[reports_df["true_label"] == "first"].sample(n=10, random_state=42)
sample_q2 = reports_df[reports_df["true_label"] == "second"].sample(n=10, random_state=42)
sample_q3 = reports_df[reports_df["true_label"] == "third"].sample(n=10, random_state=42)
sample_q4 = reports_df[reports_df["true_label"] == "fourth"].sample(n=10, random_state=42)

sample_df = pd.concat([sample_q1, sample_q2, sample_q3, sample_q4], ignore_index=True)
sample_df = sample_df.sample(frac=1.0, random_state=42).reset_index(drop=True)

sample_df[["id", "report_type", "filed_date", "true_label"]].head()

/home/maxim-shibanov/Projects_Py/Risk-and-return-prediction-with-LLM/src/data_analysis/data_fetcher/data_fetcher_class.py:111: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)
/home/maxim-shibanov/Projects_Py/Risk-and-return-prediction-with-LLM/src/data_analysis/data_fetcher/data_fetcher_class.py:92: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)
/home/maxim-shibanov/Projects_Py/Risk-and-return-prediction-with-LLM/src/data_analysis/data_fetcher/data_fetcher_class.py:130: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2

Available regressors:
 - avg_default_verbolizer
 - avg_shrink_verbolizer
 - doc_len
 - eps_surprise
 - f_size
 - full_list_default_verbolizer
 - full_list_shrink_verbolizer
 - hv_orig_score
 - lm_orig_score
 - max_abs_default
 - max_abs_shrink
 - max_default_verbolizer
 - max_shrink_verbolizer
 - md_hv1
 - md_hv2
 - md_hv3
 - md_lm1
 - md_lm2
 - md_lm3
 - min_default_verbolizer
 - min_shrink_verbolizer
 - stretch_default
 - stretch_shrink
Available sectors:
 - Technology (92)
 - Industrials (86)
 - Financial Services (85)
 - Healthcare (66)
 - Consumer Cyclical (58)
 - Consumer Defensive (40)
 - Real Estate (32)
 - Utilities (32)
 - Energy (30)
 - Basic Materials (23)
 - Communication Services (22)


,id,report_type,filed_date,true_label
0,88361,10-Q,2022-04-29,second
1,26105,10-Q,2019-04-22,second
2,22304,10-Q,2019-05-03,second
3,30847,10-Q,2019-07-23,third
4,103015,10-K,2023-02-24,first


In [5]:
def evaluate_prompt(prompt: str, data: pd.DataFrame) -> tuple[dict, pd.DataFrame]:
    rows = []
    for row in tqdm(data.itertuples(index=False), total=len(data), desc=prompt[:48]):
        result = model_driver.predict_metadata(
            verbolizer=fq_verbolizer,
            prompt=prompt,
            text=row.raw_text,
        )
        probabilities = result["probabilities"]
        predicted_label = max(probabilities, key=probabilities.get)
        rows.append(
            {
                "id": row.id,
                "report_type": row.report_type,
                "filed_date": row.filed_date,
                "true_label": row.true_label,
                "predicted_label": predicted_label,
                "confidence": result["confidence"],
                "true_label_probability": probabilities[row.true_label],
            }
        )

    predictions_df = pd.DataFrame(rows)
    y_true = predictions_df["true_label"]
    y_pred = predictions_df["predicted_label"]

    metrics = {
        "prompt": prompt,
        "avg_confidence": predictions_df["confidence"].mean(),
        "avg_true_label_probability": predictions_df["true_label_probability"].mean(),
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
    }
    return metrics, predictions_df


all_metrics = []
all_predictions = {}

for prompt in quarter_prompts:
    metrics, predictions_df = evaluate_prompt(prompt, sample_df)
    all_metrics.append(metrics)
    all_predictions[prompt] = predictions_df

    print("\nPROMPT:", prompt)
    print(predictions_df["predicted_label"].value_counts())
    print(pd.crosstab(predictions_df["true_label"], predictions_df["predicted_label"]))

results_df = pd.DataFrame(all_metrics)
results_df = results_df.sort_values(["f1_macro", "avg_confidence"], ascending=[False, False]).reset_index(drop=True)
display(results_df)

Read the report excerpt and identify the calenda: 100%|██████████| 40/40 [01:42<00:00,  2.57s/it]



PROMPT: Read the report excerpt and identify the calendar quarter of the SEC filing date. Selected quarter:
predicted_label
fourth    14
third     13
first      9
second     4
Name: count, dtype: int64
predicted_label  first  fourth  second  third
true_label                                   
first                0      10       0      0
fourth               0       1       0      9
second               8       2       0      0
third                1       1       4      4


Classify the calendar quarter of the filing date: 100%|██████████| 40/40 [01:41<00:00,  2.54s/it]



PROMPT: Classify the calendar quarter of the filing date for this SEC report. Classification:
predicted_label
fourth    31
first      6
third      3
Name: count, dtype: int64
predicted_label  first  fourth  third
true_label                           
first                1       8      1
fourth               1       7      2
second               2       8      0
third                2       8      0


You are tagging SEC filings by submission quarte: 100%|██████████| 40/40 [01:40<00:00,  2.51s/it]


...

Based on the reporting language and report struc: 100%|██████████| 40/40 [01:37<00:00,  2.44s/it]



PROMPT: Based on the reporting language and report structure, assign this filing to the company's first, second, third, or fourth reporting quarter of the year. Company quarter:
predicted_label
fourth    24
first     16
Name: count, dtype: int64
predicted_label  first  fourth
true_label                    
first                3       7
fourth               3       7
second               6       4
third                4       6


You are labeling reports by quarter order within: 100%|██████████| 40/40 [01:37<00:00,  2.44s/it]


PROMPT: You are labeling reports by quarter order within each company's reporting year. From this excerpt, the most likely company quarter rank is:
predicted_label
first     30
third      6
fourth     3
second     1
Name: count, dtype: int64
predicted_label  first  fourth  second  third
true_label                                   
first                7       2       0      1
fourth               5       1       0      4
second               9       0       1      0
third                9       0       0      1


,prompt,avg_confidence,avg_true_label_probability,accuracy,precision_macro,recall_macro,f1_macro
0,You are labeling reports by quarter order with...,0.047536,0.244551,0.250,0.433333,0.250,0.202666
1,You are tagging company reports by quarter ran...,0.074653,0.216991,0.200,0.194498,0.200,0.195627
2,Imagine you are reviewing SEC metadata and nee...,0.125270,0.218845,0.200,0.196825,0.200,0.194675
3,Determine the quarter number of this report wi...,0.097246,0.257047,0.225,0.150000,0.225,0.175000
4,Imagine you are reviewing company reporting me...,0.071755,0.207243,0.175,0.175267,0.175,0.173888
5,Based on the reporting language and report str...,0.105318,0.239647,0.250,0.119792,0.250,0.160633
6,Read the report excerpt and identify this comp...,0.078333,0.212463,0.150,0.149432,0.150,0.149206
7,Treat this as a metadata classification task f...,0.136723,0.219166,0.150,0.145563,0.150,0.144345
8,Ignore the SEC submission month and focus on t...,0.114110,0.224349,0.150,0.141270,0.150,0.142043
9,Treat this as a metadata classification task f...,0.061226,0.238143,0.200,0.104688,0.200,0.116667


In [6]:
top_3_prompts = results_df.head(3).copy()
display(top_3_prompts[["prompt", "avg_confidence", "precision_macro", "recall_macro", "f1_macro", "accuracy"]])

for idx, row in top_3_prompts.iterrows():
    print(f"\nTop {idx + 1} prompt")
    print(row["prompt"])
    print(
        {
            "avg_confidence": row["avg_confidence"],
            "precision_macro": row["precision_macro"],
            "recall_macro": row["recall_macro"],
            "f1_macro": row["f1_macro"],
            "accuracy": row["accuracy"],
        }
    )

,prompt,avg_confidence,precision_macro,recall_macro,f1_macro,accuracy
0,You are labeling reports by quarter order with...,0.047536,0.433333,0.25,0.202666,0.25
1,You are tagging company reports by quarter ran...,0.074653,0.194498,0.20,0.195627,0.20
2,Imagine you are reviewing SEC metadata and nee...,0.125270,0.196825,0.20,0.194675,0.20



Top 1 prompt
You are labeling reports by quarter order within each company's reporting year. From this excerpt, the most likely company quarter rank is:
{'avg_confidence': 0.047536181285977364, 'precision_macro': 0.43333333333333335, 'recall_macro': 0.24999999999999997, 'f1_macro': 0.20266608391608393, 'accuracy': 0.25}

Top 2 prompt
You are tagging company reports by quarter rank within the company year. From this report excerpt, the company quarter is
{'avg_confidence': 0.07465294197318144, 'precision_macro': 0.19449786324786322, 'recall_macro': 0.2, 'f1_macro': 0.19562674802949404, 'accuracy': 0.2}

Top 3 prompt
Imagine you are reviewing SEC metadata and need to assign the filing to a calendar quarter. The correct quarter is
{'avg_confidence': 0.12526982141425833, 'precision_macro': 0.19682539682539682, 'recall_macro': 0.2, 'f1_macro': 0.1946749226006192, 'accuracy': 0.2}


In [7]:
# Optional: inspect one prediction with top-token printing.
example_prompt = top_3_prompts.iloc[0]["prompt"]
example_report = sample_df.iloc[0]["raw_text"]
model_driver.predict_metadata(
    verbolizer=fq_verbolizer,
    prompt=example_prompt,
    text=example_report,
    print_top_tokens=True,
    top_k_tokens=20,
)


Top 20 tokens by probability:
Quarter      | 0.035400
quarter      | 0.032227
From         | 0.007172
March        | 0.005951
Date         | 0.005768
Third        | 0.005585
label        | 0.005096
Ex           | 0.004791
is           | 0.003967
within       | 0.003723
in           | 0.003616
January      | 0.003616
as           | 0.003494
April        | 0.003494
date         | 0.003296
Order        | 0.002991
Label        | 0.002899
please       | 0.002411
Last         | 0.002197
Within       | 0.002197


{'confidence': 0.03538346290588379,
 'probabilities': {'first': 0.3364991341495462,
  'third': 0.2758222210243314,
  'fourth': 0.23689601034977664,
  'second': 0.15078263447634577}}